# Sparkle V2 — Fase 3.6e: ensemble de proyecciones

**Protocolo:** `Protocolo_Fase36_Camino_al_70.md` **Enmienda 4** (2026-07-29).

## Qué corrige

§14.3 mostró que la proyección aleatoria de 1.280 → 16 es un **parámetro molesto con σ ≈ 2 pts**
que ningún IC bootstrap captura (el bootstrap remuestrea participantes, no matrices). Por la
regla 5, ningún número basado en un solo sorteo es reportable:

| | Un sorteo | Media ± σ sobre sorteos |
| :--- | ---: | ---: |
| Backbone actual, acc c=0,65 | 0,6919 (bloque 0) | **0,6714 ± 0,0199** (0/8 ≥ 0,70) |
| Backbone nuevo, acc c=0,70 | 0,7009 (semilla 42) | **0,6762 ± 0,0230** (1/5 ≥ 0,70) |

## Configuración primaria — sin ningún sorteo nuevo

Backbone **actual** (el desplegado). Las 8 proyecciones son los **8 bloques disjuntos de 16
columnas** de la matriz de 128 que ya existe. El ensemble agota el artefacto que el proyecto ya
tiene: **cero decisiones arbitrarias nuevas**, y reproducible bit a bit por un tercero.

Se ajusta una cabeza logística por bloque y se **promedian las probabilidades**.

**Despliegue:** la extensión ya calcula las 128 dimensiones. Son 8 cabezas lineales de 96
entradas = 776 parámetros (contra 321 del modelo A). No cambia extractor ni preprocesamiento.

**El Test no se toca.**

In [ ]:
import sys, os, json, collections, time
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')

CACHE_32  = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
CACHE_OLD = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_5_raw1280.npz'
CACHE_V2  = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_6_raw_v2.npz'
PROJ_ORIG = f'{PROJECT_DIR}/checkpoints_fase3_2/proyeccion_aleatoria_original.npz'
OOF_OLD   = f'{PROJECT_DIR}/fase3_5_proyeccion/oof_predicciones.npz'
OUT_36    = f'{PROJECT_DIR}/fase3_6_camino70'
os.makedirs(OUT_36, exist_ok=True)

import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# ------- CONGELADOS -------
SEED, UMBRAL, N_FOLDS = 42, 0.50, 5
D_PROJ, N_PROY, K, SPAN_MAX = 16, 8, 5, 7
COBERTURAS = np.round(np.arange(1.00, 0.499, -0.05), 2)
ACC_OBJETIVO, COB_MIN, BALACC_PISO = 0.70, 0.60, 0.5896
N_BOOT, N_FRAMES, DIM_RAW = 5000, 60, 1280
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen', 'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']

d32 = np.load(CACHE_32, allow_pickle=True)
dold = np.load(CACHE_OLD, allow_pickle=True)
Wo = np.load(PROJ_ORIG); W_ORIG, B_ORIG = Wo['W'], Wo['b']
assert W_ORIG.shape == (DIM_RAW, D_PROJ * N_PROY), W_ORIG.shape
zold = np.load(OOF_OLD, allow_pickle=True)
FEATS22 = [str(f) for f in d32['feature_names']]
ids_of = lambda a: np.array([str(c) for c in a])

Xg_pool = np.concatenate([d32['Xg_tr'], d32['Xg_va']])
y3_pool = np.concatenate([d32['y3_tr'], d32['y3_va']])
id_pool = np.concatenate([ids_of(d32['id_tr']), ids_of(d32['id_va'])])
y_pool = (y3_pool > 0).astype(np.int64)
part_pool = np.array([c[:6] for c in id_pool])
print(f'Pool {len(y_pool)} clips · {len(set(part_pool))} participantes · '
      f'matriz original {W_ORIG.shape} → {N_PROY} bloques de {D_PROJ}')

## 1. Utilidades — idénticas a 3.5b/3.6b/3.6d

In [ ]:
def interpolate_sequence(seq):
    df = pd.DataFrame(seq).interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean(X, medians=None, thresh=0.30):
    keep, out = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > thresh:
            continue
        s = interpolate_sequence(X[i])
        if np.isnan(s).any() and medians is not None:
            idx = np.where(np.isnan(s)); s[idx] = np.take(medians, idx[1])
        out.append(s); keep.append(i)
    return np.stack(out), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X):
    d_ = np.zeros_like(X[:, :, delta_idx]); d_[:, 1:, :] = np.diff(X[:, :, delta_idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

def resumir(X):
    X = np.nan_to_num(X); return np.hstack([X.mean(axis=1), X.std(axis=1)])

def metricas(y, p, umbral=UMBRAL):
    pred = (p > umbral).astype(int); triv = max((y == 0).mean(), (y == 1).mean())
    acc = accuracy_score(y, pred)
    return {'n': int(len(y)), 'trivial': round(float(triv), 4),
            'accuracy': round(float(acc), 4), 'brecha': round(float(acc - triv), 4),
            'balanced_acc': round(float(balanced_accuracy_score(y, pred)), 4),
            'f1_macro': round(float(f1_score(y, pred, average='macro')), 4),
            'auc': round(float(roc_auc_score(y, p)), 4)}

def _ba(y, ok):
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    return np.nan if (n0 == 0 or n1 == 0) else 0.5*(ok[y==0].sum()/n0 + ok[y==1].sum()/n1)

## 2. Bucle OOF con ensemble

**La limpieza se calcula UNA vez por fold** y se reutiliza para las 8 proyecciones — no depende
de la proyección. (En 3.6d se recalculaba 8 veces y por eso tardaba ~55 min.)

In [ ]:
def correr_oof_ensemble(Xa_pool, W, b, n_proy=N_PROY, d=D_PROJ):
    '''Devuelve (p_ens, P_ind, mask): ensemble y las n_proy predicciones individuales.'''
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    P_ind = np.full((n_proy, len(y_pool)), np.nan)
    mask = np.zeros(len(y_pool), bool)
    for fold, (tr, va) in enumerate(sgkf.split(Xg_pool, y_pool, groups=part_pool)):
        assert not (set(part_pool[tr]) & set(part_pool[va])), f'LEAKAGE fold {fold}'  # regla 7
        # ---- limpieza y geometría: UNA sola vez por fold ----
        g_med = np.nanmedian(Xg_pool[tr].reshape(-1, 22), axis=0)
        Xg_a, kga = clean(Xg_pool[tr], g_med); Xg_b, kgb = clean(Xg_pool[va], g_med)
        Ea, kea = clean(Xa_pool[tr].astype(np.float32), None, thresh=0.5)
        Eb, keb = clean(Xa_pool[va].astype(np.float32), None, thresh=0.5)
        ka = np.intersect1d(kga, kea); kb = np.intersect1d(kgb, keb)
        ig = {k: i for i, k in enumerate(kga)}; ie = {k: i for i, k in enumerate(kea)}
        jg = {k: i for i, k in enumerate(kgb)}; je = {k: i for i, k in enumerate(keb)}
        gsc = StandardScaler().fit(add_deltas(Xg_a).reshape(-1, 32))
        sg = lambda X: gsc.transform(X.reshape(-1, 32)).reshape(len(X), N_FRAMES, 32)
        Rg_tr = resumir(sg(add_deltas(Xg_a))[[ig[k] for k in ka]])
        Rg_ev = resumir(sg(add_deltas(Xg_b))[[jg[k] for k in kb]])
        Ea_k = Ea[[ie[k] for k in ka]]; Eb_k = Eb[[je[k] for k in kb]]
        y_tr = y_pool[tr][ka]; idx = va[kb]; mask[idx] = True
        # ---- una cabeza por proyección ----
        for j in range(n_proy):
            cols = slice(j*d, (j+1)*d)
            R_tr = np.hstack([Rg_tr, resumir(Ea_k @ W[:, cols] + b[cols])])
            R_ev = np.hstack([Rg_ev, resumir(Eb_k @ W[:, cols] + b[cols])])
            sc = StandardScaler().fit(R_tr)
            lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
            lr.fit(sc.transform(R_tr), y_tr)
            P_ind[j, idx] = lr.predict_proba(sc.transform(R_ev))[:, 1]
        print(f'  fold {fold} listo ({len(idx)} clips × {n_proy} proyecciones)')
    assert not np.isnan(P_ind[:, mask]).any()
    return P_ind[:, mask].mean(axis=0), P_ind[:, mask], mask

Xa_old = np.concatenate([dold['Xr_tr'], dold['Xr_va']])
t0 = time.time()
p_ens, P_ind, mask = correr_oof_ensemble(Xa_old, W_ORIG, B_ORIG)
print(f'Ensemble OOF listo en {(time.time()-t0)/60:.1f} min')

# Alinear con el OOF de referencia
ids_new = id_pool[mask]; ids_old = np.array([str(c) for c in zold['ids']])
assert set(ids_new) == set(ids_old), 'El subconjunto OOF cambió'
orden = {c: i for i, c in enumerate(ids_new)}
sel = np.array([orden[c] for c in ids_old])
p_ens = p_ens[sel]; P_ind = P_ind[:, sel]
y_c = zold['y']; part_c = np.array([str(p) for p in zold['part']]); ids_c = ids_old
print(f'✅ {len(ids_c)} clips alineados')

## 3. Nivel clip — ensemble vs. proyecciones individuales

In [ ]:
ind = [metricas(y_c, P_ind[j])['balanced_acc'] for j in range(N_PROY)]
print(f'Proyecciones individuales (clip): BalAcc {np.mean(ind):.4f} ± {np.std(ind):.4f} '
      f'| rango [{min(ind):.4f}, {max(ind):.4f}]')
print(f'ENSEMBLE (clip): {metricas(y_c, p_ens)}')

def boot_pareado(pa, pb, y, part, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); parts = np.unique(part)
    idx = {q: np.where(part == q)[0] for q in parts}
    oka = (pa > UMBRAL).astype(int) == y; okb = (pb > UMBRAL).astype(int) == y
    d = np.empty(n_boot)
    for i in range(n_boot):
        j = np.concatenate([idx[q] for q in rng.choice(parts, size=len(parts), replace=True)])
        d[i] = _ba(y[j], okb[j]) - _ba(y[j], oka[j])
    d = d[~np.isnan(d)]; lo, hi = np.percentile(d, [2.5, 97.5])
    return {'delta_balacc': round(float(d.mean()), 4),
            'IC95': [round(float(lo), 4), round(float(hi), 4)], 'cruza_cero': bool(lo <= 0 <= hi)}

print('\nEnsemble vs. bloque 0 (el reportado en §13):',
      boot_pareado(P_ind[0], p_ens, y_c, part_c))

## 4. Ventanas y curva riesgo-cobertura

In [ ]:
def parse(cid):
    c = os.path.splitext(str(cid).strip())[0]; return c[:6], int(c[6:])

def ventanas_con_huecos(ids, k=K, span_max=SPAN_MAX):
    por = collections.defaultdict(list)
    for i, cid in enumerate(ids):
        s, t = parse(cid); por[s].append((t, i))
    wins = []
    for s, pares in por.items():
        pares.sort(); i = 0
        while i + k - 1 < len(pares):
            v = pares[i:i+k]
            if v[-1][0] - v[0][0] <= span_max - 1:
                wins.append([j for _, j in v]); i += k
            else:
                i += 1
    return wins

wins = ventanas_con_huecos(ids_c)
for w in wins:
    assert len({parse(ids_c[i])[0] for i in w}) == 1
    ts = sorted(parse(ids_c[i])[1] for i in w); assert ts[-1]-ts[0] <= SPAN_MAX-1
print(f'{len(wins)} ventanas · cobertura {sum(len(w) for w in wins)/len(ids_c):.1%}')

def agregar(wins, y, p, ids):
    return (np.array([int(y[w].sum()*2 > len(w)) for w in wins]),
            np.array([float(p[w].mean()) for w in wins]),
            np.array([parse(ids[w[0]])[0] for w in wins]))

def curva(p, y):
    conf = np.abs(p - UMBRAL); orden = np.argsort(-conf); filas = []
    for c in COBERTURAS:
        k = max(1, int(round(c*len(p)))); s = orden[:k]
        filas.append({'cobertura': float(c), **metricas(y[s], p[s])})
    return pd.DataFrame(filas)

y_w, pw_ens, part_w = agregar(wins, y_c, p_ens, ids_c)
df_ens = curva(pw_ens, y_w)
print('\n=== VENTANA · ENSEMBLE ==='); print(df_ens.to_string(index=False))

# Las 8 individuales, para la comparación honesta contra §14.3
filas_ind = []
PW = []
for j in range(N_PROY):
    _, pw_j, _ = agregar(wins, y_c, P_ind[j], ids_c); PW.append(pw_j)
    d_j = curva(pw_j, y_w)
    filas_ind.append({'bloque': j,
                      'acc_c065': d_j[d_j.cobertura == 0.65].accuracy.iloc[0],
                      'acc_c060': d_j[d_j.cobertura == 0.60].accuracy.iloc[0]})
dfi = pd.DataFrame(filas_ind)
print(f'\nIndividuales acc c=0,65: {dfi.acc_c065.mean():.4f} ± {dfi.acc_c065.std():.4f} '
      f'(§14.3 reportó 0,6714 ± 0,0199)')
print(f'ENSEMBLE acc c=0,65: {df_ens[df_ens.cobertura==0.65].accuracy.iloc[0]:.4f}')

pw_mejor_ind = PW[int(np.argmax(dfi.acc_c065))]
print('\nEnsemble vs. MEJOR individual (ventana, c=1,00):',
      boot_pareado(pw_mejor_ind, pw_ens, y_w, part_w))
print('Ensemble vs. bloque 0 (ventana, c=1,00):',
      boot_pareado(PW[0], pw_ens, y_w, part_w))

## 5. IC clusterizado y veredicto

In [ ]:
def boot_cob(p, y, part, c, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); parts = np.unique(part)
    idx = {q: np.where(part == q)[0] for q in parts}; conf = np.abs(p - UMBRAL)
    acc, ba = np.empty(n_boot), np.empty(n_boot)
    for i in range(n_boot):
        j = np.concatenate([idx[q] for q in rng.choice(parts, size=len(parts), replace=True)])
        j = j[np.argsort(-conf[j])[:max(1, int(round(c*len(j))))]]
        ys = y[j]; ok = ((p[j] > UMBRAL).astype(int) == ys)
        acc[i] = ok.mean(); ba[i] = _ba(ys, ok)
    pct = lambda v: [round(float(x), 4) for x in np.percentile(v[~np.isnan(v)], [2.5, 97.5])]
    return {'accuracy_IC95': pct(acc), 'balacc_IC95': pct(ba)}

for c in (1.00, 0.70, 0.65, 0.60):
    f = df_ens[df_ens.cobertura == c].iloc[0]; r = boot_cob(pw_ens, y_w, part_w, c)
    print(f'c={c:.2f} | acc {f.accuracy:.4f} {r["accuracy_IC95"]} | '
          f'BalAcc {f.balanced_acc:.4f} {r["balacc_IC95"]} | trivial {f.trivial:.4f} | '
          f'brecha {f.brecha:+.4f}')

plt.figure(figsize=(7, 4))
plt.plot(df_ens.cobertura, df_ens.accuracy, 'o-', label='accuracy ENSEMBLE')
plt.plot(dfi.bloque*0 + COBERTURAS[COBERTURAS == 0.65][0], dfi.acc_c065, 'x',
         color='gray', alpha=.6, label='individuales (c=0,65)')
plt.plot(df_ens.cobertura, df_ens.trivial, 's--', alpha=.5, label='trivial')
plt.axhline(ACC_OBJETIVO, color='r', ls=':', label='objetivo 0,70')
plt.axvline(COB_MIN, color='gray', ls=':')
plt.gca().invert_xaxis(); plt.legend(fontsize=8); plt.xlabel('cobertura')
plt.tight_layout(); plt.savefig(f'{OUT_36}/ensemble.png', dpi=120); plt.show()

print('================ VEREDICTO PASO 4 (ENSEMBLE) ================')
cands = []
for _, f in df_ens[df_ens.cobertura >= COB_MIN].iterrows():
    if f.accuracy >= ACC_OBJETIVO:
        r = boot_cob(pw_ens, y_w, part_w, f.cobertura)
        if r['balacc_IC95'][0] >= BALACC_PISO:
            cands.append({'cobertura': f.cobertura, 'accuracy': f.accuracy,
                          'trivial': f.trivial, 'brecha': f.brecha,
                          'balanced_acc': f.balanced_acc, **r})
if not cands:
    m = df_ens[df_ens.cobertura >= COB_MIN].sort_values('accuracy').iloc[-1]
    print(f'❌ No alcanza. Mejor admisible: {m.accuracy:.4f} a cobertura {m.cobertura:.2f} '
          f'(BalAcc {m.balanced_acc:.4f}, brecha {m.brecha:+.4f}).')
    print('   Fase 3.6 queda agotada. Cerrar con la evidencia acumulada.')
else:
    e = sorted(cands, key=lambda r: -r['cobertura'])[0]
    print('✅ Cumple. Se elige la MAYOR cobertura que satisface:')
    print(json.dumps(e, indent=2, ensure_ascii=False))
    print('\n⚠️  Sexta decisión sobre OOF. Test vendrá por debajo y su IC medirá ~±7 pts con 19 '
          'participantes. Congelar TODO y escribir la enmienda de cierre antes de abrirlo.')
    print('⚠️  A diferencia de §13/§14, este número NO depende de un sorteo: usa los 8 bloques '
          'de la matriz desplegada. Es reproducible bit a bit.')

np.savez(f'{OUT_36}/oof_ensemble.npz', p_ens=p_ens, P_ind=P_ind, y=y_c, part=part_c, ids=ids_c)
json.dump({'individuales_clip': ind, 'curva_ensemble': df_ens.to_dict('records'),
           'individuales_ventana': dfi.to_dict('records'), 'candidatas': cands},
          open(f'{OUT_36}/paso4_ensemble.json', 'w'), indent=2, ensure_ascii=False, default=str)
print('\nEvaluaciones sobre Test consumidas: 0.')

## 6. Secundario descriptivo — el mismo ensemble sobre el backbone nuevo

**No se usa para elegir.** El paso 3 no encontró efecto demostrable del backbone (§14.4) y la
Enmienda 4 fija el actual como primario. Esto se corre solo para dejarlo registrado.

In [ ]:
CORRER_SECUNDARIO = False     # ponerlo en True solo para completar el registro
if CORRER_SECUNDARIO and os.path.exists(CACHE_V2):
    dv2 = np.load(CACHE_V2, allow_pickle=True)
    Xa_v2 = np.concatenate([dv2['Xr_tr'], dv2['Xr_va']])
    rng = np.random.default_rng(2000)
    W2 = rng.normal(0, 1/np.sqrt(DIM_RAW), size=(DIM_RAW, D_PROJ*N_PROY)).astype(np.float32)
    p2, _, m2 = correr_oof_ensemble(Xa_v2, W2, np.zeros(D_PROJ*N_PROY, np.float32))
    p2 = p2[sel]
    _, pw2, _ = agregar(wins, y_c, p2, ids_c)
    print('\n=== VENTANA · ENSEMBLE backbone NUEVO (secundario) ===')
    print(curva(pw2, y_w).to_string(index=False))
    print('Nuevo − actual (ensemble, ventana):', boot_pareado(pw_ens, pw2, y_w, part_w))
else:
    print('Secundario no ejecutado (CORRER_SECUNDARIO = False).')